In [8]:
import pandas as pd
from datetime import datetime
from pathlib import Path
import sys
import holidays

ROOT = Path().resolve().parents[1] 
sys.path.append(str(ROOT))

from DWH.connection.connect import loadIN

In [9]:
# bereik
start_date = '2010-01-01'
end_date = '2027-01-01'

# Basis range aanmaken
dates = pd.date_range(start=start_date, end=end_date)
df_date = pd.DataFrame({'fullDate_alternate_key': dates})

In [10]:
df_date.head()

,fullDate_alternate_key
0,2010-01-01
1,2010-01-02
2,2010-01-03
3,2010-01-04
4,2010-01-05


In [11]:
# Kolommen toevoegen conform de SQL tabel (PascalCase)
df_date['DateKey'] = df_date['fullDate_alternate_key'].dt.strftime('%Y%m%d').astype(int)
df_date['FullDateAlternateKey'] = df_date['fullDate_alternate_key'].dt.date
df_date['DayOfMonth'] = df_date['fullDate_alternate_key'].dt.day
df_date['EnglishDayNameOfWeek'] = df_date['fullDate_alternate_key'].dt.day_name()

# Nederlandse vertaling
dutch_days = {
    'Monday': 'maandag', 'Tuesday': 'dinsdag', 'Wednesday': 'woensdag', 
    'Thursday': 'donderdag', 'Friday': 'vrijdag', 'Saturday': 'zaterdag', 'Sunday': 'zondag'
}
df_date['DutchDayNameOfWeek'] = df_date['EnglishDayNameOfWeek'].map(dutch_days)

df_date['DayOfWeek'] = df_date['fullDate_alternate_key'].dt.dayofweek + 1
df_date['DayOfWeekInMonth'] = (df_date['DayOfMonth'] - 1) // 7 + 1
df_date['DayOfWeekInYear'] = df_date['fullDate_alternate_key'].dt.dayofyear
df_date['DayOfQuarter'] = (df_date['fullDate_alternate_key'] - df_date['fullDate_alternate_key'].dt.to_period('Q').dt.start_time).dt.days + 1
df_date['DayOfYear'] = df_date['fullDate_alternate_key'].dt.dayofyear

df_date['WeekOfMonth'] = (df_date['DayOfMonth'] - 1) // 7 + 1
df_date['WeekOfQuarter'] = ((df_date['DayOfYear'] - 1) // 7) % 13 + 1
df_date['WeekOfYear'] = df_date['fullDate_alternate_key'].dt.isocalendar().week.astype(int)

df_date['Month'] = df_date['fullDate_alternate_key'].dt.month
df_date['EnglishMonthName'] = df_date['fullDate_alternate_key'].dt.month_name()

dutch_months = {
    1: 'januari', 2: 'februari', 3: 'maart', 4: 'april', 5: 'mei', 6: 'juni', 
    7: 'juli', 8: 'augustus', 9: 'september', 10: 'oktober', 11: 'november', 12: 'december'
}
df_date['DutchMonthName'] = df_date['Month'].map(dutch_months)

df_date['MonthOfQuarter'] = (df_date['Month'] - 1) % 3 + 1
df_date['Quarter'] = df_date['fullDate_alternate_key'].dt.quarter
df_date['QuarterName'] = 'Q' + df_date['Quarter'].astype(str)
df_date['Year'] = df_date['fullDate_alternate_key'].dt.year
df_date['MonthYear'] = df_date['fullDate_alternate_key'].dt.strftime('%m-%Y')
df_date['MMYYYY'] = df_date['fullDate_alternate_key'].dt.strftime('%m%Y')

In [12]:
# holidays toevoegen
years = list(range(2010, datetime.now().year + 1))
df_date['IsHoliday'] = df_date['FullDateAlternateKey'].isin(holidays.Belgium(years=years))
df_date['HolidayName'] = df_date['FullDateAlternateKey'].map(holidays.Belgium(years=years))

In [13]:
df_final_date = df_date.drop(columns=['fullDate_alternate_key'])

print(df_final_date['HolidayName'].unique())

<StringArray>
[          'Nieuwjaar',                   nan,               'Pasen',
         'Paasmaandag',   'Dag van de Arbeid', 'O. L. H. Hemelvaart',
          'Pinksteren',     'Pinkstermaandag',  'Nationale feestdag',
 'O. L. V. Hemelvaart',       'Allerheiligen',      'Wapenstilstand',
            'Kerstmis']
Length: 13, dtype: str


In [14]:
loadIN(table='DimDate', df=df_final_date)

'data has been loaded into table DimDate in database DEPI within the localhost server'